### Config

In [1]:
import re

In [2]:
DATA_PATH = "data/short_story.txt"

### Read text

In [3]:
text = None
with open(DATA_PATH, 'r') as f:
    text = f.read()

In [4]:
text[:100]+'[...]'

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g[...]'

In [5]:
len(text)

20479

### Simple splitting

In [6]:
tokens = re.split(r'([,.?_!"()\']|--|\s)', text)
tokens = [item.strip() for item in tokens if item.strip()]
print(len(tokens))

4649


In [7]:
tokens[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

In [8]:
vocab_flat = sorted(set(tokens))
vocab_flat[:25]

['!',
 '"',
 "'",
 '(',
 ')',
 ',',
 '--',
 '.',
 ':',
 ';',
 '?',
 'A',
 'Ah',
 'Among',
 'And',
 'Are',
 'Arrt',
 'As',
 'At',
 'Be',
 'Begin',
 'Burlington',
 'But',
 'By',
 'Carlo']

In [9]:
len(vocab_flat)

1159

In [10]:
vocab = {i: w for (i,w) in enumerate(vocab_flat)}

In [11]:
list(vocab.items())[:25]

[(0, '!'),
 (1, '"'),
 (2, "'"),
 (3, '('),
 (4, ')'),
 (5, ','),
 (6, '--'),
 (7, '.'),
 (8, ':'),
 (9, ';'),
 (10, '?'),
 (11, 'A'),
 (12, 'Ah'),
 (13, 'Among'),
 (14, 'And'),
 (15, 'Are'),
 (16, 'Arrt'),
 (17, 'As'),
 (18, 'At'),
 (19, 'Be'),
 (20, 'Begin'),
 (21, 'Burlington'),
 (22, 'But'),
 (23, 'By'),
 (24, 'Carlo')]

### Tokenizer implementation

In [12]:
class RegexTokenizer:
    
    UNK = "<UNK>"
    END_OF_TEXT = "<EOS>"
    
    def __init__(self, split_regex=r'([,.?_!"()\']|--|\s)'):
        self.vocab = None
        self.tok_to_id = None
        self.id_to_tok = None
        self.split_regex = split_regex

    def fit(self, text:str):
        tokens = self._split(text)
        self.vocab = sorted(set(tokens))
        self.tok_to_id =  {w:i for (i,w) in enumerate(self.vocab)}
        self.id_to_tok = {i:w for (w,i) in self.tok_to_id.items()}
        self._add_special_token(self.UNK)
        self._add_special_token(self.END_OF_TEXT)
        
    def encode(self, text: str) -> list[int]:
        tokens = self._split(text)
        unk_id = self.tok_to_id[self.UNK]
        return [self.tok_to_id.get(tok, unk_id) for tok in tokens]

    def decode(self, token_ids: list[int]) -> list[str]:
        return [self.id_to_tok[i] for i in token_ids]

    def _split(self, text:str) -> list[str]:
        tokens = re.split(self.split_regex, text)
        tokens = [item.strip() for item in tokens if item.strip()]
        return tokens

    def _add_special_token(self, token:str):
        tok_id = len(self.tok_to_id)
        self.tok_to_id[token] = tok_id
        self.id_to_tok[tok_id] = token

### Tokenizer usage

In [13]:
tokenizer = RegexTokenizer()
tokenizer.fit(text)

In [14]:
cutoff=24
print(text[:cutoff])
tok_ids = tokenizer.encode(text[:cutoff])
tok_ids

I HAD always thought Jac


[55, 46, 154, 1028, 1159]

In [15]:
tokenizer.decode(tok_ids)

['I', 'HAD', 'always', 'thought', '<UNK>']

In [16]:
sent1 = "Hello, do you like tea?"
sent2 = "In the sunlit terraces of the palace."
sents = sent1 + f' {tokenizer.END_OF_TEXT} ' + sent2
sents

'Hello, do you like tea? <EOS> In the sunlit terraces of the palace.'

In [17]:
' '.join(tokenizer.decode(tokenizer.encode(sents)))

'<UNK> , do you like tea ? <EOS> In the sunlit terraces of the <UNK> .'